## WHO Health Inequality Capstone

#### MODULE 1 — Data Ingestion and Understanding

In [1]:
import pandas as pd
import numpy as np

In [2]:
xl = pd.ExcelFile('WHO Dataset.xlsx')
print("Sheet names:", xl.sheet_names)

Sheet names: ['Data', 'About']


#### Reading About the Sheet First

In [3]:
about = pd.read_excel('WHO Dataset.xlsx', sheet_name='About', header=None)
for line in about[0]:
    print(line)

WHO Health Inequality Data Repository
Sustainable Development Goals (SDGs)
Metadata available at: https://www.who.int/data/sets/health-inequality-monitor-dataset#sdg


#### Loading the Actual Dataset

In [4]:
df = pd.read_excel('WHO Dataset.xlsx', sheet_name = 'Data')
print('Shape:', df.shape)

Shape: (443150, 24)


#### Listing Every Column

In [5]:
print(list(df.columns))

['setting', 'date', 'source', 'indicator_abbr', 'indicator_name', 'dimension', 'subgroup', 'estimate', 'se', 'ci_lb', 'ci_ub', 'population', 'flag', 'setting_average', 'iso3', 'favourable_indicator', 'indicator_scale', 'ordered_dimension', 'subgroup_order', 'reference_subgroup', 'whoreg6', 'wbincome2025', 'dataset_id', 'update']


#### Checking Data Types

In [6]:
print(df.dtypes)

setting                  object
date                      int64
source                   object
indicator_abbr           object
indicator_name           object
dimension                object
subgroup                 object
estimate                float64
se                      float64
ci_lb                   float64
ci_ub                   float64
population              float64
flag                     object
setting_average         float64
iso3                     object
favourable_indicator      int64
indicator_scale           int64
ordered_dimension         int64
subgroup_order            int64
reference_subgroup        int64
whoreg6                  object
wbincome2025             object
dataset_id               object
update                   object
dtype: object


In [7]:
df.head(5)

,setting,date,source,indicator_abbr,indicator_name,dimension,subgroup,estimate,se,ci_lb,...,iso3,favourable_indicator,indicator_scale,ordered_dimension,subgroup_order,reference_subgroup,whoreg6,wbincome2025,dataset_id,update
0,Albania,2002,UN SDG Indicators Database,SI_POV_EMP1,1.1.1 Employed population below international ...,Age (2 groups) (15-25+),15-24 years,0.48,NaN,NaN,...,ALB,0,100,0,0,0,European,Upper-middle-income,rep_sdg,15 September 2025
1,Albania,2002,UN SDG Indicators Database,SI_POV_EMP1,1.1.1 Employed population below international ...,Age (2 groups) (15-25+),25+ years,0.50,NaN,NaN,...,ALB,0,100,0,0,1,European,Upper-middle-income,rep_sdg,15 September 2025
2,Albania,2002,UN SDG Indicators Database,SI_POV_EMP1,1.1.1 Employed population below international ...,Sex,Female,0.48,NaN,NaN,...,ALB,0,100,0,0,0,European,Upper-middle-income,rep_sdg,15 September 2025
3,Albania,2002,UN SDG Indicators Database,SI_POV_EMP1,1.1.1 Employed population below international ...,Sex,Male,0.51,NaN,NaN,...,ALB,0,100,0,0,1,European,Upper-middle-income,rep_sdg,15 September 2025
4,Albania,2005,UN SDG Indicators Database,SI_POV_EMP1,1.1.1 Employed population below international ...,Age (2 groups) (15-25+),15-24 years,0.62,NaN,NaN,...,ALB,0,100,0,0,0,European,Upper-middle-income,rep_sdg,15 September 2025


#### Understanding the Row-Level Grain

In [8]:
grain_duplicates = df.duplicated(subset=['setting', 'date', 'indicator_abbr', 'dimension', 'subgroup']).sum()
print("Duplicate grain combinations:", grain_duplicates)

Duplicate grain combinations: 0


##### This proves the "grain" of the dataset — the precise definition of what one row represents. Here, one row = one country, in one year, for one indicator, showing one subgroup's value within one dimension.

#### Counting Key Entities

In [9]:
print("Unique countries (setting):", df['setting'].nunique())
print("Unique indicators (indicator_abbr):", df['indicator_abbr'].nunique())
print("Year range (date):", df['date'].min(), "to", df['date'].max())
print("Unique dimensions:", df['dimension'].nunique())

Unique countries (setting): 198
Unique indicators (indicator_abbr): 211
Year range (date): 1998 to 2025
Unique dimensions: 24


#### Confirming Source and Dataset Metadata

In [10]:
print("source unique values:", df['source'].unique())
print("dataset_id unique values:", df['dataset_id'].unique())

source unique values: ['UN SDG Indicators Database']
dataset_id unique values: ['rep_sdg']


#### Document the Sex Dimension Specifically
##### As this is our primary Scope

In [13]:
sex_df = df[df['dimension'] == 'Sex']
print('Total Sex-dimension rows:', len(sex_df))
print(sex_df['subgroup'].value_counts())

Total Sex-dimension rows: 161026
subgroup
Female    80513
Male      80513
Name: count, dtype: int64


#### Understanding the reference subgroup for Sex

In [14]:
print(sex_df.groupby('subgroup')['reference_subgroup'].unique())

subgroup
Female    [0]
Male      [1]
Name: reference_subgroup, dtype: object


### MODULE 2 — Data Cleaning and Quality Validation

#### Checking Missing values

In [15]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
audit = pd.DataFrame({'null_count': null_counts,'null_pct': null_pct})
print(audit)

                      null_count  null_pct
setting                        0      0.00
date                           0      0.00
source                         0      0.00
indicator_abbr                 0      0.00
indicator_name                 0      0.00
dimension                      0      0.00
subgroup                       0      0.00
estimate                   26800      6.05
se                        443150    100.00
ci_lb                     406457     91.72
ci_ub                     406212     91.66
population                443150    100.00
flag                           0      0.00
setting_average            97587     22.02
iso3                           0      0.00
favourable_indicator           0      0.00
indicator_scale                0      0.00
ordered_dimension              0      0.00
subgroup_order                 0      0.00
reference_subgroup             0      0.00
whoreg6                      132      0.03
wbincome2025                5132      1.16
dataset_id 

#### Removing the Two Fully-Empty Columns as they serve no purpose 

In [16]:
df_clean = df.drop(columns = ['population','se'])
print('Shape before:',df.shape)
print('Shape after:',df_clean.shape)

Shape before: (443150, 24)
Shape after: (443150, 22)


#### Finding the Missing WHO Regions

In [21]:
missing_region = df[df['whoreg6'].isnull()]
print("Rows with missing whoreg6:", len(missing_region))
print("Which settings do these belong to?")
print(missing_region['setting'].unique())

Rows with missing whoreg6: 132
Which settings do these belong to?
['Faroe Islands']


### Finding Duplicates

In [23]:
exact_duplicates = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


### Removing Any Duplicates Programmatically (As Safety Net)
#### Why we're doing it because this is a defensive coding habit, not a reaction to a problem. 
#### If WHO ever updates this dataset and accidentally introduces a duplicate row in a future version, this line silently protects your pipeline without you needing to remember to re-check manually. Confirmed here: 0 rows removed, exactly as expected.

In [25]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Shape: {df_clean.shape}")

Removed 0 duplicate rows. Shape: (443150, 22)


In [31]:
print("estimate range:", df_clean['estimate'].min(), "to", df_clean['estimate'].max())
print("date range:", df_clean['date'].min(), "to", df_clean['date'].max())
print("indicator_scale unique values:", sorted(df_clean['indicator_scale'].unique()))
print("Negative estimates:", (df_clean['estimate'] < 0).sum())
print('\nChecks the minimum and maximum of the key numeric columns, and confirms no `estimate` value is negative') 
print('which would be logically invalid for the kinds of indicators in this dataset — as we cannot have a negative mortality rate or negative percentage')

estimate range: 0.0 to 323952.21
date range: 1998 to 2025
indicator_scale unique values: [np.int64(2), np.int64(100), np.int64(1000), np.int64(10000), np.int64(100000)]
Negative estimates: 0

Checks the minimum and maximum of the key numeric columns, and confirms no `estimate` value is negative
which would be logically invalid for the kinds of indicators in this dataset — as we cannot have a negative mortality rate or negative percentage


In [32]:
df_clean['date'] = df_clean['date'].astype(int)
df_clean['estimate'] = df_clean['estimate'].astype(float)
print(df_clean[['date','estimate']].dtypes)

date          int64
estimate    float64
dtype: object


### Final Shape Check and Save

In [37]:
print("Final cleaned shape:", df_clean.shape)
print("Columns remaining:", len(df_clean.columns))
print("Rows unchanged from original:", df_clean.shape[0] == df.shape[0])

df_clean.to_csv(r'who_data_cleaned.csv', index=False)
print("Saved to who_data_cleaned.csv")

Final cleaned shape: (443150, 22)
Columns remaining: 22
Rows unchanged from original: True
Saved to Users\MONKIT\Desktop\Data Science\Capstone Project\Datasets/who_data_cleaned.csv
